# ============================================================
# 1. ONE-HOT ENCODING (word-level)
# ============================================================
Aim: Convert each unique word into a binary vector.

In [1]:

from sklearn.preprocessing import OneHotEncoder
document = ["my name is sunny and I love AI"]

# Tokenize: lowercase + split into words
tokens = document[0].lower().split()

# OneHotEncoder expects a 2D input -> wrap each word in its own list
words_2d = [[word] for word in tokens]

encoder = OneHotEncoder(sparse_output=False)
encoder.fit(words_2d)

print("Vocabulary:", encoder.categories_[0])

encoded = encoder.transform(words_2d)
for word, vec in zip(tokens, encoded):
    print(word, "->", vec)

Vocabulary: ['ai' 'and' 'i' 'is' 'love' 'my' 'name' 'sunny']
my -> [0. 0. 0. 0. 0. 1. 0. 0.]
name -> [0. 0. 0. 0. 0. 0. 1. 0.]
is -> [0. 0. 0. 1. 0. 0. 0. 0.]
sunny -> [0. 0. 0. 0. 0. 0. 0. 1.]
and -> [0. 1. 0. 0. 0. 0. 0. 0.]
i -> [0. 0. 1. 0. 0. 0. 0. 0.]
love -> [0. 0. 0. 0. 1. 0. 0. 0.]
ai -> [1. 0. 0. 0. 0. 0. 0. 0.]


# ============================================================
# 2. BAG OF WORDS (BoW) — CountVectorizer
# ============================================================
Aim: Represent documents as word-frequency counts.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
documents = [
    "people watch movie and watch movie again",
    "people watch cricket and watch cricket",
    "people like movie and like movie a lot",
    "people like cricket"
]

bow = CountVectorizer()
bow_matrix = bow.fit_transform(documents)

print("\nBoW Vocabulary:", bow.get_feature_names_out())
print("BoW Matrix:\n", bow_matrix.toarray())

# Transform a brand-new sentence using the already-fitted vocabulary
new_doc = ["lion is the king of jungle"]
print("New doc encoded:\n", bow.transform(new_doc).toarray())


BoW Vocabulary: ['again' 'and' 'cricket' 'like' 'lot' 'movie' 'people' 'watch']
BoW Matrix:
 [[1 1 0 0 0 2 1 2]
 [0 1 2 0 0 0 1 2]
 [0 1 0 2 1 2 1 0]
 [0 0 1 1 0 0 1 0]]
New doc encoded:
 [[0 0 0 0 0 0 0 0]]


# ============================================================
# 3. TF-IDF — TfidfVectorizer
# ============================================================
 Aim: Weight words by importance (frequency in doc vs across all docs).

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
documents = [
    "people watch movie and watch movie again",
    "people watch cricket and watch cricket",
    "people like movie and like movie a lot",
    "people like cricket"
]
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

print("\nTF-IDF Vocabulary:", tfidf.get_feature_names_out())
print("TF-IDF Matrix:\n", tfidf_matrix.toarray())


TF-IDF Vocabulary: ['again' 'and' 'cricket' 'like' 'lot' 'movie' 'people' 'watch']
TF-IDF Matrix:
 [[0.38771136 0.24747114 0.         0.         0.         0.61135218
  0.20232387 0.61135218]
 [0.         0.2684707  0.66322944 0.         0.         0.
  0.21949239 0.66322944]
 [0.         0.24747114 0.         0.61135218 0.38771136 0.61135218
  0.20232387 0.        ]
 [0.         0.         0.64043405 0.64043405 0.         0.
  0.42389674 0.        ]]


In [4]:
!uv pip install -q gensim

In [5]:
"""
Classical Word Embeddings — Simplified
Covers: Pretrained Word2Vec (Google News, 300-dim), similarity/analogy
        operations, and Average Word2Vec for sentence-level vectors.
"""

import numpy as np
import gensim.downloader as api


# ============================================================
# 1. LOAD PRETRAINED WORD2VEC MODEL
# ============================================================
# Trained on Google News data. Each word -> 300-dimensional vector.
# NOTE: ~1.6GB download the first time it's called.
model = api.load("word2vec-google-news-300")


# ============================================================
# 2. WORD VECTORS
# ============================================================
print(model["sunny"])          # 300-dim vector for a single word
print(len(model["sunny"]))     # 300

# NOTE: model[...] only works on a SINGLE known word, not a phrase.
# model["i am sunny"] would raise a KeyError — a model must be tokenized
# and each word looked up individually (see Average Word2Vec below).


# ============================================================
# 3. SIMILARITY & ANALOGY OPERATIONS
# ============================================================
print(model.most_similar("man"))
print(model.most_similar("cricket"))
print(model.most_similar("sunny"))

print(model.similarity("man", "woman"))   # ~0.77 -> closely related
print(model.similarity("man", "python"))  # ~0.21 -> unrelated
print(model.similarity("man", "man"))     # 1.0   -> identical
print(model.similarity("happy", "joy"))   # ~0.36
print(model.similarity("python", "java")) # ~0.13

# Case-sensitive: "sunny" and "Sunny" are different tokens
print(model.similarity("sunny", "Sunny"))

# Odd-one-out
print(model.doesnt_match(["PHP", "JAVA", "DOG", "C++"]))  # -> 'DOG'

# Classic analogy: king - man + woman = queen
vec = model["king"] - model["man"] + model["woman"]
print(model.most_similar([vec]))


# ============================================================
# 4. AVERAGE WORD2VEC (sentence -> single vector)
# ============================================================
# Idea: get a vector for every word in the sentence, then average
# them to get one fixed-size vector representing the whole sentence.

def average_word2vec(sentence: str, model) -> np.ndarray:
    """Convert a sentence into a single averaged Word2Vec vector."""
    words = sentence.lower().split()
    word_vectors = [model[word] for word in words if word in model]
    if not word_vectors:
        raise ValueError("None of the words in the sentence are in the vocabulary.")
    return np.mean(word_vectors, axis=0)


sentence = "I love machine learning"
sentence_vector = average_word2vec(sentence, model)

print(sentence_vector)
print(sentence_vector.shape)   # (300,)

[==========================------------------------] 53.7% 893.1/1662.8MB downloaded

#SOTA Techniques

In [13]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = os.getenv("token_read_hf", "")
google_api= userdata.get('gemini_key')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

# ====================================================================
# 1. HUGGING FACE — open-source, runs locally, free
# ====================================================================

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

hf_model = SentenceTransformer("all-MiniLM-L6-v2")

word ="machine"
sentence = "How to reduce heart disease risk?"
paragraph = """
Machine learning is a field of artificial intelligence that focuses on building systems
that learn from data. It is widely used in applications like recommendation systems,
image recognition, and natural language processing.
"""

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
word_embedding = hf_model.encode(word)
sentence_embedding = hf_model.encode(sentence)
paragraph_embedding = hf_model.encode(paragraph)

print("Word embedding shape:", word_embedding.shape)
print("Sentence embedding shape:", sentence_embedding.shape)
print("Paragraph embedding shape:", paragraph_embedding.shape)

Word embedding shape: (384,)
Sentence embedding shape: (384,)
Paragraph embedding shape: (384,)


In [ ]:
word_embedding

In [5]:
similarity_word_sentence = cosine_similarity([word_embedding], [sentence_embedding])
similarity_word_paragraph = cosine_similarity([word_embedding], [paragraph_embedding])

print("Similarity between word and sentence:", similarity_word_sentence[0][0])
print("Similarity between word and paragraph:", similarity_word_paragraph[0][0])

Similarity between word and sentence: -0.0112975
Similarity between word and paragraph: 0.40449536



# ====================================================================
# 2. GEMINI — Google's embedding API (via LangChain)
# ================================================================

In [7]:
!pip install -q langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 18.6 MB/s eta 0:00:00


In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",api_key=google_api)

gemini_embedding = gemini_model.embed_query("What is the capital of France?")
print("Gemini (gemini-embedding-001) dimension:", len(gemini_embedding))  # 3072

Gemini (gemini-embedding-001) dimension: 3072




# ====================================================================
# 3. OPENAI — embedding API (via LangChain)
# ================================================================

In [ ]:
# comming soon